# Iris 分類器 - Rust 実装チュートリアル

このノートブックでは、Rust で実装した Iris 分類器のコードを解説します。

**注意**: Windows 環境では evcxr_jupyter (Rust カーネル) のビルドに問題があるため、このノートブックは Python カーネルを使用してコードを説明します。実際の Rust コードは `examples/` ディレクトリと `src/models/` ディレクトリにあります。

## 1. プロジェクト構造

```
app/rust/
├── src/
│   ├── lib.rs              # ライブラリルート
│   ├── error.rs            # エラー型定義
│   └── models/
│       ├── mod.rs          # モデルモジュール
│       └── iris.rs         # Iris 分類器の実装
├── examples/
│   ├── iris_train.rs       # 訓練スクリプト
│   └── iris_validate.rs    # 交差検証スクリプト
└── data/
    └── iris.csv            # Iris データセット
```

## 2. エラーハンドリング

Rust では Result 型を使った堅牢なエラーハンドリングが可能です。

### src/error.rs

```rust
use thiserror::Error;

pub type Result<T> = std::result::Result<T, Error>;

#[derive(Error, Debug)]
pub enum Error {
    #[error("IO error: {0}")]
    Io(#[from] std::io::Error),

    #[error("CSV error: {0}")]
    Csv(#[from] csv::Error),

    #[error("Model error: {0}")]
    Model(String),

    #[error("Unknown species: {0}")]
    UnknownSpecies(String),
}
```

**特徴**:
- `thiserror` クレートで簡潔にエラー型を定義
- `#[from]` 属性で他のエラー型からの自動変換
- 各エラーに明確なメッセージ

## 3. Iris 分類器の実装

### 3.1 構造体定義

```rust
use linfa::prelude::*;
use linfa_trees::DecisionTree;
use ndarray::{Array1, Array2};

pub struct IrisClassifier {
    model: Option<DecisionTree<f64, usize>>,
}
```

**ポイント**:
- `Option<DecisionTree>` で未訓練状態を表現
- `DecisionTree<f64, usize>` は特徴量が f64、ターゲットが usize

## 4. 実行方法

### 4.1 訓練スクリプトの実行

```bash
# ターミナルで実行
cd app/rust
cargo run --example iris_train
```

出力例:
```
=== Iris 分類モデルの訓練 ===

1. データ読み込み中...
   データ件数: 150 件
   特徴量数: 4 個
   訓練データ: 120 件
   検証データ: 30 件

2. モデル訓練中...
   訓練完了

3. モデル評価:
   訓練データ精度: 98.33%
   検証データ精度: 70.00%
```

## 5. Python からの実行（参考）

Rust のコードを Python から実行したい場合、以下のように subprocess を使用できます：

In [ ]:
import subprocess
import os

# カレントディレクトリを app/rust に変更
rust_dir = os.path.abspath('..')
os.chdir(rust_dir)

print("現在のディレクトリ:", os.getcwd())
print("\n訓練スクリプトを実行中...\n")

# cargo run --example iris_train を実行
result = subprocess.run(
    ['cargo', 'run', '--example', 'iris_train'],
    capture_output=True,
    text=True
)

print(result.stdout)
if result.stderr:
    print("エラー:", result.stderr)

## 6. Rust vs Python の比較

### 6.1 型安全性

**Rust:**
```rust
// コンパイル時に型チェック
let classifier = IrisClassifier::new();
let (features, targets) = classifier.load_data(&path)?;  // Result<T, E>
```

**Python:**
```python
# 実行時エラーの可能性
classifier = IrisClassifier()
features, targets = classifier.load_data(path)  # 例外の可能性
```

### 6.2 パフォーマンス

**Rust:**
- ゼロコスト抽象化
- コンパイル時最適化
- メモリ安全性の保証

**Python:**
- インタープリタ言語
- NumPy/pandas で高速化
- GIL による並列処理の制限

## 7. まとめ

### Rust で機械学習を実装するメリット:

1. **型安全性**: コンパイル時にバグを検出
2. **パフォーマンス**: C/C++ 並みの実行速度
3. **メモリ安全性**: 所有権システムでメモリリークを防止
4. **並行処理**: データ競合のない安全な並列処理
5. **エラーハンドリング**: Result 型で明示的なエラー処理

### 学習リソース:

- [The Rust Book](https://doc.rust-lang.org/book/)
- [linfa ドキュメント](https://rust-ml.github.io/linfa/)
- [ndarray ドキュメント](https://docs.rs/ndarray/)

### 次のステップ:

1. Chapter 5: Cinema 回帰モデルの実装
2. Chapter 6: Survived 分類器の実装
3. Chapter 7: Boston 回帰モデルの実装
4. Chapter 8: Web API の構築